# Analytics vs. numerics: all seven QLT methods agree
levisim solves the linearised Heisenberg-Langevin equations of a cavity-levitated array seven ways: the linear amplifier model (LAM) and its Hybrid and TNT variants, the Numerical and Power Series hybridisation methods, and the two closed-form analytic methods of the mechanical-mode-comb paper (PRR 8; Analytic Inside and Analytic Output). This notebook measures how closely every method reproduces LAM, then overlays the analytic and numerical spectra with unequal tweezer phases, the case the closed form was derived for.

Result: every method reproduces LAM to within a relative difference of 9.1 × 10⁻¹⁴ on the default systems (three arrays on the node with unequal tweezer phases and one 3D bead off the node); the largest differences are those of the two analytic methods for the x₁z₁ cross spectrum of the off-node bead. With `FULL_TABLE = True` (adding 4×2 and a 2×3 off-node array) the largest is 4.9 × 10⁻¹³ (Power Series H, 2×3 off-node).

Runtime: about 20 s (`FULL_TABLE = True`: several times longer).

# START

NOTES:
- made for MMC paper plots https://doi.org/10.48550/arXiv.2406.00366 (published in PRR 8, where Fig. 4 compares the closed-form Eq. (17) with the numerics)
- the analytic methods were generalised in November 2026 (levisim PRs #22-#24): the closed form now includes the direct mechanical coupling between the axes of one bead (g_xy, g_xz, g_yz), and P and the thermal brackets are evaluated in a form that does not cancel catastrophically when the frequency grid hits a resonance exactly
- measured on levisim main at ce13c89 (November 2026): over the default table below, the largest relative difference from LAM is 3.3 × 10⁻¹⁵ for Hybrid LAM, 1.4 × 10⁻¹⁴ for TNT LAM, 5.0 × 10⁻¹⁴ for Numerical H, 6.8 × 10⁻¹⁴ for Power Series H and 9.1 × 10⁻¹⁴ for both analytic methods (3.6 × 10⁻¹⁴ on the node with unequal phases). With `FULL_TABLE = True` the analytic methods stay within 4.8 × 10⁻¹⁴ on the 4×2 array; on the 2×3 off-node array every non-LAM method is 1.9-4.9 × 10⁻¹³ from LAM, but there the full-matrix methods (LAM, Hybrid LAM) agree with each other to 2.4 × 10⁻¹⁵ and the reduced methods (Numerical H, both analytic) to 8.8 × 10⁻¹⁴ in the homodyne spectrum, so the spread is round-off of order 10⁻¹³ between the two kinds of calculation, not a fault of one method

# Agreement of all seven methods with LAM
For each system and spectrum, the cell below computes the largest relative difference from the linear amplifier model (LAM, the full numerical solution of the Heisenberg-Langevin equations by matrix inversion) over the whole frequency grid,

$$\epsilon = \max_\omega \frac{|S_\mathrm{method}(\omega) - S_\mathrm{LAM}(\omega)|}{|S_\mathrm{LAM}(\omega)|},$$

for the homodyne and heterodyne spectra, every position PSD and the cross spectrum of the first and last axis. The grid is `DEFAULT_ω_range` (250-320 kHz in 0.05 kHz steps), which lands exactly on every mechanical frequency, where the old analytic closed form used to lose precision.

Systems: the x-y arrays on the cavity node with unequal tweezer phases (`PHASE_DEPENDENT_xy_dict`, φₙ = 0°, 30°, 60°), and one bead off the node in 3D (`OFF_NODE_toy_params_pairs`: z coupling, intra-bead couplings g_xy, g_xz, g_yz). TNT LAM ignores input-output theory, so it gives position PSDs only; Power Series H runs where its terms are cheap (at most 4 dimensions by default; `FULL_TABLE = True` adds it for 6 dimensions, plus the 4×2 and 2×3 systems, about 7 min).

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from levisim import *
from levisim.multiphysics.utils.shelf import (
    PHASE_DEPENDENT_xy_dict,
    OFF_NODE_toy_params_pairs,
    DEFAULT_gas_kwargs,
    DEFAULT_cavity_kwargs,
    DEFAULT_ω_range,
)

set_plt_rcParams_to_xmgrace_style()

# True adds Power Series H for 6 dimensions and the 4×2 and 2×3 systems (about 7 min in total;
# the 4×2 system has no power series terms: 8! = 40,320 determinant terms are not stored)
FULL_TABLE = False

LAM = QuantumLinearTheoryMethod.LINEAR_AMPLIFIER_MODEL
TNT_LAM = QuantumLinearTheoryMethod.TNT_LINEAR_AMPLIFIER_MODEL
POWER_SERIES_H = QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION
other_methods = [
    QuantumLinearTheoryMethod.HYBRID_LINEAR_AMPLIFIER_MODEL,
    TNT_LAM,
    QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION,
    POWER_SERIES_H,
    QuantumLinearTheoryMethod.ANALYTIC_INSIDE_CAVITY,
    QuantumLinearTheoryMethod.ANALYTIC_CAVITY_OUTPUT,
]


def off_node_mtrap(system):
    """One of the off-node toy systems (z axis, intra-bead couplings, 0°/45° tweezer phases)"""
    N, d = (int(_) for _ in system.split('×'))
    return MultiTrap(
        num_traps=N,
        Cartesian_axes_per_trap=d,
        ω_range=DEFAULT_ω_range,
        gas_kwargs=DEFAULT_gas_kwargs,
        cavity_kwargs=DEFAULT_cavity_kwargs,
        toy_params=OFF_NODE_toy_params_pairs,
        generate_power_series=FULL_TABLE or N * d <= 4,
    )


systems = {
    '1×2 node': PHASE_DEPENDENT_xy_dict['1×2'][-1],
    '2×2 node': PHASE_DEPENDENT_xy_dict['2×2'][-1],
    '3×2 node': PHASE_DEPENDENT_xy_dict['3×2'][-1],
    '1×3 off-node': off_node_mtrap('1×3'),
}
if FULL_TABLE:
    systems['4×2 node'] = PHASE_DEPENDENT_xy_dict['4×2'][-1]
    systems['2×3 off-node'] = off_node_mtrap('2×3')


def run_power_series(mtrap):
    """Power Series H needs the series terms (loaded or generated); by default only up to 4 dimensions"""
    return hasattr(mtrap, 'deranged_determinant_by_order') and (FULL_TABLE or mtrap.num_dimensions <= 4)


def spectra(mtrap, method):
    """{spectrum name: array} for the given method (TNT LAM has position PSDs only)"""
    label_A, label_B = mtrap.labels[0], mtrap.labels[-1]
    values = {'PSDs': mtrap.all_position_power_spectra(method, zpf_scaled=True)[0]}
    if method != TNT_LAM:
        values['homodyne'] = mtrap.homodyne_spectrum(method)
        values['heterodyne'] = mtrap.heterodyne_spectrum(method)
        values[f'CSD {label_A},{label_B}'] = mtrap.position_cross_spectrum(label_A, label_B, method)
    return values


def max_relative_difference(values, reference):
    values, reference = np.asarray(values), np.asarray(reference)
    return np.max(np.abs(values - reference) / np.abs(reference))


ϵ = {}  # {(system, spectrum): {method: max relative difference from LAM}}
LAM_values = {}
seconds = {}
for system, mtrap in systems.items():
    t0 = time.time()
    LAM_values[system] = spectra(mtrap, LAM)
    for method in other_methods:
        if method == POWER_SERIES_H and not run_power_series(mtrap):
            continue
        for spectrum, values in spectra(mtrap, method).items():
            ϵ.setdefault((system, spectrum), {})[method] = max_relative_difference(values, LAM_values[system][spectrum])
    seconds[system] = time.time() - t0

# table
columns = list(ϵ)
print('\n' + ', '.join(f'{system}: {t:.0f} s' for system, t in seconds.items()))
print(f'{"max relative difference from LAM":34s}' + ''.join(f'{s:>14s}' for s, _ in columns))
print(f'{"":34s}' + ''.join(f'{sp[:13]:>14s}' for _, sp in columns))
for method in other_methods:
    row = [ϵ[column].get(method) for column in columns]
    print(f'{repr(method):34s}' + ''.join(f'{v:14.1e}' if v is not None else f'{"-":>14s}' for v in row))

worst_value, worst_method, worst_column = max(((ϵ[c][m], m, c) for c in columns for m in ϵ[c]), key=lambda item: item[0])
analytic = [ϵ[c][m] for c in columns for m in ϵ[c] if 'ANALYTIC' in m.name]
print(f'\nlargest relative difference from LAM: {worst_value:.1e} ({worst_method!r}, {worst_column[0]}, {worst_column[1]})')
print(f'largest for the two analytic methods: {max(analytic):.1e}')

# heat map of log10(ϵ)
matrix = np.array([[ϵ[c].get(m, np.nan) for c in columns] for m in other_methods])
fig, ax = plt.subplots(figsize=(LATEX_FIG_WIDTH * 1.8, 3.2), dpi=100)
# exact agreement (ϵ = 0) is drawn at the bottom of the colour scale and labelled 0
image = ax.imshow(np.log10(np.maximum(matrix, 1e-17)), cmap='viridis', vmin=-16, vmax=-12, aspect='auto')
ax.set_yticks(range(len(other_methods)), [repr(m) for m in other_methods])
ax.set_xticks(range(len(columns)), [f'{s}\n{sp}' for s, sp in columns], rotation=90, fontsize=8)
for (row, col), value in np.ndenumerate(matrix):
    if np.isfinite(value):
        ax.text(col, row, f'{np.log10(value):.1f}' if value > 0 else '0', ha='center', va='center', fontsize=6, color='w')
fig.colorbar(image, ax=ax, label=r'$\log_{10}\,\epsilon$')
ax.set_title('Largest relative difference from LAM over the frequency grid')
plt.show()

# where in frequency do the differences sit? the 2×2 homodyne spectrum, every method
system = '2×2 node'
mtrap = systems[system]
fig, ax = plt.subplots(figsize=(LATEX_FIG_WIDTH, 3.5), dpi=100)
for method in other_methods:
    if method == TNT_LAM or (method == POWER_SERIES_H and not run_power_series(mtrap)):
        continue
    values = mtrap.homodyne_spectrum(method)
    relative = np.abs(values - LAM_values[system]['homodyne']) / np.abs(LAM_values[system]['homodyne'])
    ax.semilogy(mtrap.ω_range / kHz, np.maximum(relative, 1e-17), lw=0.8, label=repr(method))
ax.vlines([dim.Ω / kHz for dim in mtrap.dimensions.values()], 1e-17, 1e-12, color=black, alpha=0.2, ls='--', label=r'all $\Omega_j$')
ax.set_xlim(mtrap.ω_range.min() / kHz, mtrap.ω_range.max() / kHz)
ax.set_ylim(1e-17, 1e-12)
ax.set_xlabel('Frequency [kHz]')
ax.set_ylabel('Relative difference from LAM')
ax.set_title(f'{system}: homodyne spectrum')
ax.legend(loc='upper left', prop=dict(size=7), ncol=2)
plt.show()


# Optical spectra

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from levisim import *
# PHASE_DEPENDENT_xy_dict is built lazily, so `from levisim import *` does not bring it in
from levisim.multiphysics.utils.shelf import PHASE_DEPENDENT_xy_dict

set_plt_rcParams_to_xmgrace_style()

for analytic_method in [
    QuantumLinearTheoryMethod.ANALYTIC_INSIDE_CAVITY,
    # QuantumLinearTheoryMethod.ANALYTIC_CAVITY_OUTPUT,
]:
    for numeric_method in [
        QuantumLinearTheoryMethod.LINEAR_AMPLIFIER_MODEL,
        # QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION,
    ]:
        for system in [
            '2×2',
            # '3×2',
            # '4×2',
        ]:
            for mtrap in [
                # PHASE_EQUAL_xy_dict[system][+1],
                # PHASE_EQUAL_xy_dict[system][-1],
                # PHASE_DEPENDENT_xy_dict[system][+1],
                PHASE_DEPENDENT_xy_dict[system][-1],
            ]:
                # mtrap.ω_range = np.arange(250, 320 + 0.005, 0.005) * kHz
                ω_range = mtrap.ω_range
                xlims = np.min(ω_range) / kHz, np.max(ω_range) / kHz
                ylims = 1e-1, 1e7
                for spectrum_type in [
                    SpectrumType.HOMODYNE,
                    SpectrumType.HETERODYNE,
                ]:
                    print('\n####################################################################################################')
                    msg = f'🖼️ Using {mtrap} ({mtrap.tweezer_phase_sign_convention_str}): plotting {spectrum_type=} using {numeric_method=} vs. {analytic_method=}'
                    print(msg), logger.info(msg)

                    match spectrum_type:
                        case SpectrumType.HOMODYNE:
                            func = mtrap.homodyne_spectrum
                        case SpectrumType.HETERODYNE:
                            func = mtrap.heterodyne_spectrum

                    msg = '🎛️ Phase per tweezer: φn\n' + phase_per_tweezer_string(mtrap)
                    print(msg), logger.info(msg)

                    msg = '🎛️ Relative tweezer phase differences: δφn,m\n' + relative_tweezer_phases_string(mtrap)
                    print(msg), logger.info(msg)

                    mosaic = [['1']]
                    fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
                    ax_dict = fig.subplot_mosaic(mosaic)

                    ax = ax_dict['1']
                    ax.set_xlabel('Frequency [kHz]')
                    ax.set_ylabel(f'{repr(spectrum_type)} PSD [photons/Hz]')
                    ax.set_xlim(*xlims)
                    ax.set_ylim(*ylims)
                    analytic_vals, numeric_vals = func(analytic_method), func(numeric_method)
                    ax.text(0.1, 0.3, phase_per_tweezer_string(mtrap, latex=True, uppercase=True), color=black, fontsize=14, bbox=dict(facecolor=grey, alpha=0.2, boxstyle='round,pad=0.5'), transform=ax.transAxes)
                    ax.vlines([_.Ω / kHz for _ in mtrap.dimensions.values()], *ylims, label=r'All $\Omega_j$', color=black, alpha=0.1, ls='--')
                    ax.semilogy(ω_range / kHz, analytic_vals, color=black, label=f'Analytics')
                    ax.semilogy(ω_range / kHz, numeric_vals, color=RGB_green, label=f'Numerics', lw=1, ls='dotted')
                    # hatch is for fill pattern (adjust '.' count for density)
                    ax.fill_between(ω_range / kHz, numeric_vals, hatch='......', edgecolor=purple, facecolor='none', linewidth=1, alpha=0.5)
                    ax.legend(loc='upper left')
                    plt.show()

                    analytic_resonances, analytic_resonance_peaks = find_resonances(ω_range, analytic_vals)
                    print(f'{analytic_resonances / kHz=}')
                    print(f'{analytic_resonance_peaks=}')

                    numeric_resonances, numeric_resonance_peaks = find_resonances(ω_range, numeric_vals)
                    print(f'{numeric_resonances / kHz=}')
                    print(f'{numeric_resonance_peaks=}')
                    print('----------------------------------------------------------------------------------------------------\n')



# Position power and cross spectra

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from levisim import *
# PHASE_DEPENDENT_xy_dict is built lazily, so `from levisim import *` does not bring it in
from levisim.multiphysics.utils.shelf import PHASE_DEPENDENT_xy_dict

set_plt_rcParams_to_xmgrace_style()

for analytic_method in [
    # QuantumLinearTheoryMethod.ANALYTIC_INSIDE_CAVITY,
    QuantumLinearTheoryMethod.ANALYTIC_CAVITY_OUTPUT,
]:
    for numeric_method in [
        # QuantumLinearTheoryMethod.LINEAR_AMPLIFIER_MODEL,
        QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION,
    ]:
        print(f'{numeric_method=}, {analytic_method=}')
        for system in [
            '2×2',
            # '3×2',
            # '4×2',
        ]:
            for mtrap in [
                # PHASE_EQUAL_xy_dict[system][+1],
                # PHASE_EQUAL_xy_dict[system][-1],
                # PHASE_DEPENDENT_xy_dict[system][+1],
                PHASE_DEPENDENT_xy_dict[system][-1],
            ]:
                print('\n####################################################################################################')
                msg = f'Using {mtrap} ({mtrap.tweezer_phase_sign_convention_str}): plotting {numeric_method=} vs. {analytic_method=}'
                print(msg), logger.info(msg)

                msg = '🎛️ Phase per tweezer: φn\n' + phase_per_tweezer_string(mtrap)
                print(msg), logger.info(msg)

                msg = '🎛️ Relative tweezer phase differences: δφnm\n' + relative_tweezer_phases_string(mtrap)
                print(msg), logger.info(msg)

                # mtrap.ω_range = np.arange(250, 320 + 0.005, 0.005) * kHz
                ω_range = mtrap.ω_range
                xlims = np.min(ω_range) / kHz, np.max(ω_range) / kHz
                ylims = 1e-7, 1e3

                mosaic = [['PSD'], ['CSD']]
                fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
                ax_dict = fig.subplot_mosaic(mosaic)

                label = 'x1'
                latex = rf'{label[0]}_{{{label[1:]}}}'
                ax = ax_dict['PSD']
                ax.set_xlabel('Frequency [kHz]')
                ax.set_ylabel(f'zpf-scaled ${latex}$ PSD [${latex}$ phonons/Hz]')
                ax.set_xlim(*xlims)
                ax.set_ylim(*ylims)
                analytic_vals = mtrap.position_power_spectrum(label, analytic_method)
                numeric_vals = mtrap.position_power_spectrum(label, numeric_method)
                ax.text(0.1, 0.3, phase_per_tweezer_string(mtrap, latex=True, uppercase=True), color=black, fontsize=14, bbox=dict(facecolor=grey, alpha=0.2, boxstyle='round,pad=0.5'), transform=ax.transAxes)
                ax.vlines([_.Ω / kHz for _ in mtrap.dimensions.values()], *ylims, label=r'All $\Omega_j$', color=black, alpha=0.1, ls='--')
                ax.semilogy(ω_range / kHz, analytic_vals, color=black, label=f'Analytics')
                ax.semilogy(ω_range / kHz, numeric_vals, color=RGB_green, label=f'Numerics', lw=1, ls='dotted')
                # hatch is for fill pattern (adjust '.' count for density)
                ax.fill_between(ω_range / kHz, numeric_vals, hatch='......', edgecolor=purple, facecolor='none', linewidth=1, alpha=0.5)
                ax.legend(loc='upper left')

                analytic_resonances, analytic_resonance_peaks = find_resonances(ω_range, analytic_vals)
                print(f'{analytic_resonances / kHz=}')
                print(f'{analytic_resonance_peaks=}')

                numeric_resonances, numeric_resonance_peaks = find_resonances(ω_range, numeric_vals)
                print(f'{numeric_resonances / kHz=}')
                print(f'{numeric_resonance_peaks=}')

                label_A, label_B = 'x1', 'y2'
                latex_A = rf'{label_A[0]}_{{{label_A[1:]}}}'
                latex_B = rf'{label_B[0]}_{{{label_B[1:]}}}'
                latex_AB = latex_A + latex_B
                ax = ax_dict['CSD']
                ax.set_xlabel('Frequency [kHz]')
                ax.set_ylabel(rf'Magnitude of zpf-scaled ${latex_AB}$ CSD [$\sqrt{{{latex_AB}}}$ phonons/Hz]')
                ax.set_xlim(*xlims)
                ax.set_ylim(*ylims)
                analytic_vals = mtrap.position_cross_spectrum(label_A, label_B, analytic_method)
                numeric_vals = mtrap.position_cross_spectrum(label_A, label_B, numeric_method)
                ax.text(0.1, 0.3, phase_per_tweezer_string(mtrap, latex=True, uppercase=True), color=black, fontsize=14, bbox=dict(facecolor=grey, alpha=0.2, boxstyle='round,pad=0.5'), transform=ax.transAxes)
                ax.vlines([_.Ω / kHz for _ in mtrap.dimensions.values()], *ylims, label=r'All $\Omega_j$', color=black, alpha=0.1, ls='--')
                ax.semilogy(ω_range / kHz, np.abs(analytic_vals), color=black, label=f'Analytics')
                ax.semilogy(ω_range / kHz, np.abs(numeric_vals), color=RGB_green, label=f'Numerics', lw=1, ls='dotted')
                # hatch is for fill pattern (adjust '.' count for density)
                ax.fill_between(ω_range / kHz, np.abs(numeric_vals), hatch='......', edgecolor=purple, facecolor='none', linewidth=1, alpha=0.5)
                ax.legend(loc='upper left')

                plt.show()

                print('----------------------------------------------------------------------------------------------------')



# END
